# 02 - Data Cleaning

In this notebook, we will create a cleaner version of the raw Telco Customer Churn dataset.

The goal is to prepare the data for SQL modelling and Power BI, without changing the meaning of the original dataset.

We will focus on:

- Loading the raw Excel file
- Renaming columns into clean `snake_case`
- Checking key data types
- Understanding missing values
- Creating useful helper fields
- Exporting a cleaned CSV file

## 1. Import Libraries and Load Data

We start the same way as the inspection notebook.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
raw_file = Path("../data/raw/Telco_customer_churn.xlsx")
processed_dir = Path("../data/processed")

processed_dir.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_excel(raw_file)

df_raw.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## 2. Make a Working Copy

We do not edit `df_raw` directly. Instead, we create `df` as a working copy.

This makes it easy to compare the cleaned data with the original data if something looks strange.

In [3]:
df = df_raw.copy()

## 3. Clean Column Names

The raw columns have spaces and title case, such as `Monthly Charges`.

For Python and SQL, names like `monthly_charges` are easier to work with.

This style is called `snake_case`.

In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df.columns.tolist()

['customerid',
 'count',
 'country',
 'state',
 'city',
 'zip_code',
 'lat_long',
 'latitude',
 'longitude',
 'gender',
 'senior_citizen',
 'partner',
 'dependents',
 'tenure_months',
 'phone_service',
 'multiple_lines',
 'internet_service',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'contract',
 'paperless_billing',
 'payment_method',
 'monthly_charges',
 'total_charges',
 'churn_label',
 'churn_value',
 'churn_score',
 'cltv',
 'churn_reason']

**Your notes:**

- Pick 3 columns and compare their old names to their new names.
- Why do you think this naming style is helpful for SQL?

## 4. Check Key Business Columns

Before cleaning everything, let us focus on the columns we know will matter for churn and revenue analysis.

In [5]:
key_columns = [
    "customerid",
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "tenure_months",
    "contract",
    "payment_method",
    "monthly_charges",
    "total_charges",
    "churn_label",
    "churn_value",
    "churn_score",
    "cltv",
    "churn_reason",
]

df[key_columns].head()

,customerid,gender,senior_citizen,partner,dependents,tenure_months,contract,payment_method,monthly_charges,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason
0,3668-QPYBK,Male,No,No,No,2,Month-to-month,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,Female,No,No,Yes,2,Month-to-month,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,Female,No,No,Yes,8,Month-to-month,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,Female,No,Yes,Yes,28,Month-to-month,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,Male,No,No,Yes,49,Month-to-month,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## 5. Check Data Types Again

We want to confirm that revenue and score fields are numeric.

The most important fields here are:

- `tenure_months`
- `monthly_charges`
- `total_charges`
- `churn_value`
- `churn_score`
- `cltv`

In [6]:
df[key_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       7043 non-null   object 
 1   gender           7043 non-null   object 
 2   senior_citizen   7043 non-null   object 
 3   partner          7043 non-null   object 
 4   dependents       7043 non-null   object 
 5   tenure_months    7043 non-null   int64  
 6   contract         7043 non-null   object 
 7   payment_method   7043 non-null   object 
 8   monthly_charges  7043 non-null   float64
 9   total_charges    7043 non-null   object 
 10  churn_label      7043 non-null   object 
 11  churn_value      7043 non-null   int64  
 12  churn_score      7043 non-null   int64  
 13  cltv             7043 non-null   int64  
 14  churn_reason     1869 non-null   object 
dtypes: float64(1), int64(4), object(10)
memory usage: 825.5+ KB


## 6. Convert Numeric Columns Safely

`pd.to_numeric` turns a column into numbers.

`errors="coerce"` means: if a value cannot be converted, turn it into a blank/null value instead of crashing.

This is useful when a column looks numeric but has messy text values.

In [7]:
numeric_columns = [
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "churn_value",
    "churn_score",
    "cltv",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[numeric_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tenure_months    7043 non-null   int64  
 1   monthly_charges  7043 non-null   float64
 2   total_charges    7032 non-null   float64
 3   churn_value      7043 non-null   int64  
 4   churn_score      7043 non-null   int64  
 5   cltv             7043 non-null   int64  
dtypes: float64(2), int64(4)
memory usage: 330.3 KB


## 7. Re-check Missing Values

Now that numeric fields have been converted, we check missing values again.

If a numeric-looking field had invalid text, it may now appear as missing.

In [8]:
missing_values = df.isna().sum().sort_values(ascending=False)

missing_values[missing_values > 0]

churn_reason     5174
total_charges      11
dtype: int64

**Your notes:**

- Which columns have missing values now?
- Are those missing values a problem, or do they make sense?
- Remember: `churn_reason` is expected to be missing for customers who did not churn.

## 8. Validate Churn Fields

The dataset gives us both:

- `churn_label`: human-readable Yes/No
- `churn_value`: numeric 1/0

These should agree with each other.

In [9]:
pd.crosstab(df["churn_label"], df["churn_value"], dropna=False)

churn_value,0,1
churn_label,,
No,5174,0
Yes,0,1869


**Your notes:**

- Does `Yes` match `1`?
- Does `No` match `0`?
- If they match, why is this a good validation check?

## 9. Create Helpful Business Fields

These are simple fields that will make dashboarding easier later.

We will create:

- `is_churned`: True if the customer churned
- `is_month_to_month`: True if the contract is month-to-month
- `revenue_at_risk`: monthly revenue for churned customers

Later, we can improve `revenue_at_risk` to include high-risk non-churned customers too.

In [10]:
df["is_churned"] = df["churn_label"].eq("Yes")
df["is_month_to_month"] = df["contract"].eq("Month-to-month")
df["revenue_at_risk"] = df["monthly_charges"].where(df["is_churned"], 0)

df[["customerid", "contract", "monthly_charges", "churn_label", "is_churned", "revenue_at_risk"]].head()

,customerid,contract,monthly_charges,churn_label,is_churned,revenue_at_risk
0,3668-QPYBK,Month-to-month,53.85,Yes,True,53.85
1,9237-HQITU,Month-to-month,70.70,Yes,True,70.70
2,9305-CDSKC,Month-to-month,99.65,Yes,True,99.65
3,7892-POOKP,Month-to-month,104.80,Yes,True,104.80
4,0280-XJGEX,Month-to-month,103.70,Yes,True,103.70


## 10. Quick Metric Check

Before exporting, calculate a few simple project metrics.

These should match the story we tell later in Power BI.

In [11]:
total_customers = len(df)
churned_customers = df["is_churned"].sum()
churn_rate = churned_customers / total_customers
monthly_recurring_revenue = df["monthly_charges"].sum()
monthly_revenue_at_risk = df["revenue_at_risk"].sum()
average_monthly_charge = df["monthly_charges"].mean()

summary_metrics = pd.DataFrame({
    "metric": [
        "total_customers",
        "churned_customers",
        "churn_rate",
        "monthly_recurring_revenue",
        "monthly_revenue_at_risk",
        "average_monthly_charge",
    ],
    "value": [
        total_customers,
        churned_customers,
        churn_rate,
        monthly_recurring_revenue,
        monthly_revenue_at_risk,
        average_monthly_charge,
    ],
})

summary_metrics

,metric,value
0,total_customers,7043.000000
1,churned_customers,1869.000000
2,churn_rate,0.265370
3,monthly_recurring_revenue,456116.600000
4,monthly_revenue_at_risk,139130.850000
5,average_monthly_charge,64.761692


**Your notes:**

- What is the churn rate?
- How much monthly revenue is associated with churned customers?
- Why is monthly revenue at risk different from total charges?

## 11. Export the Cleaned Dataset

We export to CSV because it is easy for SQL tools, Power BI, and GitHub readers to understand.

The original Excel file stays untouched in `data/raw`.

In [12]:
clean_file = processed_dir / "telco_customer_churn_clean.csv"

df.to_csv(clean_file, index=False)

clean_file

WindowsPath('../data/processed/telco_customer_churn_clean.csv')

## 12. Next Step

After this notebook, we will design the analytical model:

- Which fields belong in `dim_customer`?
- Which fields belong in `dim_contract`?
- Which fields belong in `dim_service`?
- Which fields belong in `dim_payment_method`?
- Which fields belong in `fact_customer_snapshot`?

That is where the project starts becoming a proper analytics product instead of only a cleaned spreadsheet.